# 2-PandasDataframe

This notebook demonstrates how to read and process a tabular datafile with the [Pandas](https://pandas.pydata.org/) dataframe library. Pandas dataframes are limited to run on a single core and data must fit into memory.

Author: Peter W. Rose (pwrose@ucsd.edu)

In [ ]:
import os
import pandas as pd
import time

If LOCAL_SCRATCH_DIR environment variable is not set, this notebook accesses the ../data directory for temporary files.

In [ ]:
DATA_DIR = os.getenv("LOCAL_SCRATCH_DIR", default="../data")
filename = os.path.join(DATA_DIR, "gene_info.tsv")
file_size = f"{os.path.getsize(filename)/1E9:.1f}"
RESULTS_DIR = "results"
os.makedirs(RESULTS_DIR, exist_ok=True)

### Setup Benchmark

The ```file_format``` parameter is used for benchmarking different file formats. 
The cell below has been [parameterized](https://papermill.readthedocs.io/en/latest/usage-parameterize.html#jupyterlab-3-0) as input parameters for [papermill](https://papermill.readthedocs.io/en/latest/index.html).

In [ ]:
file_format = "csv"

In [ ]:
start = time.time()

### Read Data

In [ ]:
# read only specified columns and rows
column_names = ["GeneID", "Symbol", "Synonyms", "description", "type_of_gene", "#tax_id", "chromosome"]
filters=[[("type_of_gene", "==", "protein-coding")]]

if file_format == "csv":
    filename = os.path.join(DATA_DIR, "gene_info.tsv")
    genes = pd.read_csv(filename, usecols=column_names, dtype=str, sep="\t")
    genes.query("type_of_gene == 'protein-coding'", inplace=True)
elif file_format == "parquet":
    filename = os.path.join(DATA_DIR, "gene_info.parquet")
    genes = pd.read_parquet(filename, columns=column_names, filters=filters)
else:
    print("invalid file format")
    
print("Filename:", filename)
    
genes.rename(columns={"#tax_id": "tax_id"}, inplace=True)

In [ ]:
# print(f"Total memory: {genes.memory_usage(deep=True).sum()/1E9:.1f} GB")
# Total memory: 17.6 GB

### Process Data

In [ ]:
genes.head()

In [ ]:
groups = genes.groupby(["tax_id"]).size().reset_index(name="count")
groups = groups.sort_values("count", ascending=False)

### Display Results

#### Number of human protein-coding genes (tax_id = 9606)

In [ ]:
groups.query("tax_id == '9606'")

#### Top 5 organisms with the most protein-coding genes

In [ ]:
groups.head()

In [ ]:
end = time.time()

In [ ]:
df = pd.DataFrame([{"cores": 1, "time": end-start, "size": file_size, "format": file_format, "dataframe": "Pandas"}])
output_file = f"2-PandasDataframe_{file_size}_{file_format}_1.csv"
df.to_csv(os.path.join(RESULTS_DIR, output_file), index=False)

In [ ]:
print(f"Pandas: {end - start:.1f} sec.")